<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [8]</a>'.</span>

# Food Delivery ETA Prediction - Model Experiments

**Notebook:** 06_model_experiments.ipynb
**Purpose:** Train and compare multiple regression algorithms

This notebook experiments with different regression models to find the best performer for food delivery time prediction.

## Model Experiment Objective

Train and compare multiple regression algorithms:
- Baseline model (Dummy/Mean)
- Linear Regression
- Tree-based model (Random Forest)
- Gradient Boosting model
- Compare using MAE, RMSE, R²
- Track experiments with MLflow

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import mlflow
import mlflow.sklearn
import joblib
from pathlib import Path

print("Libraries imported successfully")

Libraries imported successfully


/Users/ankitalokhande/Desktop/Food_Delivery_Times/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Final Feature Dataset

In [2]:
df = pd.read_csv('../data/processed/food_delivery_features.csv')
print(f"Dataset loaded: {df.shape}")

Dataset loaded: (1000, 20)


## Define Features and Target

In [3]:
target = 'Delivery_Time_min'
features = ['Distance_km', 'Preparation_Time_min', 'Courier_Experience_yrs',
           'Total_Estimated_Time', 'Distance_Preparation_Interaction',
           'Distance_Traffic_Interaction', 'Estimated_Delivery_Time',
           'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']           'Total_Estimated_Time', 'Distance_Preparation_Interaction', 
           'Distance_Traffic_Interaction', 'Estimated_Delivery_Time',
           'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']

X = df[features]
y = df[target]

print(f"Features: {len(features)}")
print(f"Target: {target}")

Features: 11
Target: Delivery_Time_min


## Train/Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

Training set: (800, 11)
Test set: (200, 11)


## Preprocessing Pipeline

In [5]:
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

print("Preprocessing pipeline created")

Preprocessing pipeline created


/var/folders/mm/93chsfl55kvcrzyvg13_c7mh0000gn/T/ipykernel_24031/2629561200.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()


## MLflow Setup

In [6]:
# Set MLflow tracking URI to project root SQLite database
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("Food_Delivery_ETA_Prediction")
print("MLflow experiment set")

2026/09/23 22:41:06 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/09/23 22:41:06 INFO mlflow.store.db.utils: Updating database tables


2026/09/23 22:41:06 INFO mlflow.tracking.fluent: Experiment with name 'Food_Delivery_ETA_Prediction' does not exist. Creating a new experiment.


MLflow experiment set


## Baseline Model

In [7]:
with mlflow.start_run(run_name="Baseline_Mean"):
    baseline_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', DummyRegressor(strategy='mean'))
    ])
    
    baseline_pipeline.fit(X_train, y_train)
    y_pred = baseline_pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_metrics({'MAE': mae, 'RMSE': rmse, 'R2': r2})
    
    print(f"Baseline - MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.3f}")

Baseline - MAE: 17.50, RMSE: 21.23, R²: -0.006


## Linear Regression

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [8]:
with mlflow.start_run(run_name="Linear_Regression"):
    lr_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', LinearRegression())
    ])
    
    lr_pipeline.fit(X_train, y_train)
    y_pred = lr_pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_metrics({'MAE': mae, 'RMSE': rmse, 'R2': r2})
    mlflow.sklearn.log_model(lr_pipeline, "model", serialization_format='cloudpickle')
    
    print(f"Linear Regression - MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.3f}")

ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## Random Forest Regressor

In [ ]:
with mlflow.start_run(run_name="Random_Forest"):
    rf_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(n_estimators=100, random_state=42))
    ])
    
    rf_pipeline.fit(X_train, y_train)
    y_pred = rf_pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_metrics({'MAE': mae, 'RMSE': rmse, 'R2': r2})
    mlflow.log_params({'n_estimators': 100, 'random_state': 42})
    mlflow.sklearn.log_model(rf_pipeline, "model", serialization_format='cloudpickle')
    
    print(f"Random Forest - MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.3f}")

## Gradient Boosting Regressor

In [ ]:
with mlflow.start_run(run_name="Gradient_Boosting"):
    gb_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', GradientBoostingRegressor(n_estimators=100, random_state=42))
    ])
    
    gb_pipeline.fit(X_train, y_train)
    y_pred = gb_pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_metrics({'MAE': mae, 'RMSE': rmse, 'R2': r2})
    mlflow.log_params({'n_estimators': 100, 'random_state': 42})
    mlflow.sklearn.log_model(gb_pipeline, "model", serialization_format='cloudpickle')
    
    print(f"Gradient Boosting - MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.3f}")

## Save Selected Model and Test Data

In [ ]:
# Save the best model (Gradient Boosting based on actual results)
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(n_estimators=100, random_state=42))
])

final_pipeline.fit(X_train, y_train)

# Save model
Path('../models').mkdir(parents=True, exist_ok=True)
joblib.dump(final_pipeline, '../models/food_delivery_eta_model.pkl')

# Save test data for evaluation
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Final model saved to: ../models/food_delivery_eta_model.pkl")
print("Test data saved for evaluation")

## Experiment Conclusions

In [ ]:
print("="*60)
print("EXPERIMENT CONCLUSIONS")
print("="*60)

print("\nMODELS COMPARED:")
print("- Baseline (Mean): Reference point")
print("- Linear Regression: Interpretable linear model")
print("- Random Forest: Nonlinear tree-based model")
print("- Gradient Boosting: Ensemble boosting model")

print("\nMLFLOW TRACKING:")
print("- All experiments logged to MLflow")
print("- Metrics, parameters, and models tracked")
print("- Reproducible experiments with random_state=42")

print("\n" + "="*60)
print("Model experiments complete. Ready for evaluation.")
print("="*60)